In [18]:
# Pull data from the UDL, look at most observed satellites and observatories, and pull state vectors for those satellites.
# Simulate (2BEOM) objects for 1 day and simulate observations from observatories

import orekit_jpype as orekit
orekit.initVM()
from orekit_jpype.pyhelpers import setup_orekit_data
setup_orekit_data()

# Span of time to pull data for
fitspan = 10  # days

import math
import datetime
import pandas as pd
import os

from org.orekit.frames import FramesFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants, IERSConventions
from org.orekit.bodies import OneAxisEllipsoid

inertial_frame = FramesFactory.getEME2000()
itrf  = FramesFactory.getITRF(IERSConventions.IERS_2010, True)
earth = OneAxisEllipsoid(Constants.WGS84_EARTH_EQUATORIAL_RADIUS,
                          Constants.WGS84_EARTH_FLATTENING,
                          itrf)
utc = TimeScalesFactory.getUTC()
initial_date = AbsoluteDate(2024, 1, 1, 0, 0, 0.0, utc)
mu = Constants.WGS84_EARTH_MU


In [19]:
file_path = 'UDL_Data.csv'
second_header = "name,lat_deg,lon_deg,alt_m"


if os.path.exists(file_path):

    # 1. Find the line index where the second table starts
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Scan for the specific header string
    try:
        split_idx = next(i for i, line in enumerate(lines) if line.strip() == second_header)
    except StopIteration:
        raise ValueError(f"Could not find header: {second_header}")

    # 2. Read Table 1 (everything up to the split index)
    # Note: we use skipfooter because pd.read_csv needs to know where to stop
    state_vectors = pd.read_csv(file_path, nrows=split_idx)

    # 3. Read Table 2 (everything from the split index down)
    observatories = pd.read_csv(file_path, skiprows=split_idx)

    # Clean up any trailing commas/empty columns if they exist in your raw data
    state_vectors = state_vectors.dropna(axis=1, how='all')
    observatories = observatories.dropna(axis=1, how='all')
    observatories = observatories.to_dict(orient='records')

    print("--- Table 1: State Vectors ---")
    print(state_vectors.head())

    print("\n--- Table 2: Locations ---")
    print(observatories)
    dataFlag = True

else:
    print("File not found.")
    dataFlag = False

--- Table 1: State Vectors ---
                          idStateVector classificationMarking  \
0  f3588608-f38a-4a43-8fd5-0068edbb4d93         U//DS-USNO-SV   
1  a4833a23-0f74-4341-b860-7ab083f20241         U//DS-USNO-SV   
2  d35b707b-ec02-49c3-83a2-ad8ee3803568         U//DS-USNO-SV   
3  d69ac98b-2e34-4896-8486-22337e455017     U//PR-OPTICALX-SV   
4  55b4ebae-5a89-4338-8756-f3b3bd775e89         U//DS-USNO-SV   

                              epoch idOnOrbit    satNo    uct          xpos  \
0  2026-05-01 20:10:33.197000+00:00     39741  39741.0  False -23594.169452   
1  2026-05-01 18:23:28.606000+00:00     35752  35752.0  False  21643.534355   
2  2026-05-01 22:17:25.912000+00:00     40105  40105.0  False   -546.418184   
3  2026-05-02 12:05:08.294681+00:00     67433  67433.0  False    762.321463   
4  2026-05-02 05:22:45.906000+00:00     30323  30323.0  False  13058.398973   

           ypos         zpos      xvel  ...            sigmaVelUVW  rms  \
0  12367.423590   700.873400

In [20]:
# -- UDL API Utilities ----------------------------------------------------------
import requests, base64, warnings, datetime
import pandas as pd
import os
from dotenv import load_dotenv

def _noverify():
    warnings.filterwarnings('ignore', message='Unverified HTTPS request')

def UDLTokenGen(username, password):
    if not all(isinstance(v, str) for v in [username, password]):
        raise TypeError("Username and password must be strings.")
    return base64.b64encode((username + ":" + password).encode()).decode("ascii")

def UDLToDatetime(t):
    return datetime.datetime.strptime(t, "%Y-%m-%dT%H:%M:%S.%fZ")

def datetimeToUDL(t):
    return t.strftime("%Y-%m-%dT%H:%M:%S.") + f"{t.microsecond:06d}Z"

def UDLQuery(token, service, params):
    """Single synchronous UDL query -> DataFrame."""
    _noverify()
    url  = f"https://unifieddatalibrary.com/udl/{service.lower()}"
    resp = requests.get(
        url,
        headers={"Authorization": "Basic " + token},
        params=params,
        verify=False,
    )
    if resp.status_code != 200:
        raise requests.exceptions.HTTPError(
            f"UDL {service} failed ({resp.status_code}): {resp.text[:300]}"
        )
    return pd.DataFrame(resp.json())

# -- Credentials ----------------------------------------------------------------
load_dotenv()
UDL_TOKEN = os.getenv("UDL_TOKEN")
if not UDL_TOKEN:
    raise ValueError("UDL_TOKEN not found. Add it to your .env file.")
print("UDL utilities loaded.")

UDL utilities loaded.


In [21]:
if dataFlag == False:

    # -- Data Pull: Top Objects + Top Observatories + State Vectors -----------------
    time_window = f">now-{int(fitspan)} days"

    # -- 1. EO observations: last fitspan days --------------------------------------
    print(f"Querying EO observations (last {fitspan} days, up to 50,000 records)...")
    obs_raw = UDLQuery(UDL_TOKEN, "eoobservation", {
        "obTime":     time_window,
        "dataMode":   "REAL",
        "maxResults": "50000",
    })
    print(f"  Retrieved {len(obs_raw):,} observations")
    print(f"  Columns: {list(obs_raw.columns)}")

    # -- 2. Top 20 most observed satellites ----------------------------------------
    top20_counts = obs_raw["satNo"].value_counts().head(20)
    top20_ids    = top20_counts.index.tolist()
    print("\nTop 20 most observed objects:")
    for rank, (sat_no, cnt) in enumerate(top20_counts.items(), 1):
        print(f"  {rank:>2}. NORAD {sat_no:>6}: {cnt:>5} obs")

    # -- 3. Top 5 observatories -----------------------------------------------------
    sensor_col = None
    for candidate in ["senName", "sensorName", "siteName", "siteId", "sensorId", "source"]:
        if candidate in obs_raw.columns:
            sensor_col = candidate
            break

    observatories = []  # list of {name, lat_deg, lon_deg, alt_m}

    if sensor_col:
        top5_sensors = obs_raw[sensor_col].value_counts().head(5)
        top5_names   = top5_sensors.index.tolist()
        print(f"\nTop 5 observatories (by '{sensor_col}'):")

        lat_col = next((c for c in obs_raw.columns if c.lower() in
                        ("senlat", "lat", "latitude", "sitelat", "senlatitude")), None)
        lon_col = next((c for c in obs_raw.columns if c.lower() in
                        ("senlon", "lon", "longitude", "sitelon", "senlongitude")), None)
        alt_col = next((c for c in obs_raw.columns if c.lower() in
                        ("senalt", "alt", "altitude", "sitealt", "elev", "elevation", "senaltitude")), None)

        for name in top5_names:
            grp = obs_raw[obs_raw[sensor_col] == name]
            cnt = len(grp)
            lat, lon, alt_m = None, None, 0.0

            if lat_col and lon_col and not grp[lat_col].dropna().empty and not grp[lon_col].dropna().empty:
                lat   = float(grp[lat_col].dropna().iloc[0])
                lon   = float(grp[lon_col].dropna().iloc[0])
                raw_a = float(grp[alt_col].dropna().iloc[0]) if (alt_col and not grp[alt_col].dropna().empty) else 0.0
                alt_m = raw_a * 1000 if raw_a < 100 else raw_a  # km->m heuristic
            else:
                # Fall back to UDL sensor service
                try:
                    sdf   = UDLQuery(UDL_TOKEN, "sensor", {"name": str(name)})
                    lat   = float(sdf["lat"].iloc[0])
                    lon   = float(sdf["lon"].iloc[0])
                    raw_a = float(sdf.get("alt", sdf.get("altitude", pd.Series([0]))).iloc[0])
                    alt_m = raw_a * 1000 if raw_a < 100 else raw_a
                except Exception as e:
                    print(f"  WARN: coords unavailable for '{name}' ({e}). Using (0,0,0).")
                    lat, lon, alt_m = 0.0, 0.0, 0.0

            observatories.append({"name": str(name), "lat_deg": lat, "lon_deg": lon, "alt_m": alt_m})
            print(f"  {name}: lat={lat:.2f} deg  lon={lon:.2f} deg  alt={alt_m/1e3:.2f} km  |  {cnt} obs")
    else:
        print("\nNo sensor column found -> using fallback observatory list.")
        observatories = [
            {"name": "Maui/MSSS",    "lat_deg":  20.71, "lon_deg": -156.26, "alt_m": 3058.0},
            {"name": "Socorro/NMSF", "lat_deg":  34.07, "lon_deg": -106.91, "alt_m": 1477.0},
            {"name": "Diego Garcia", "lat_deg":  -7.31, "lon_deg":   72.41, "alt_m":    5.0},
            {"name": "Learmonth",    "lat_deg": -22.24, "lon_deg":  114.10, "alt_m":   27.0},
            {"name": "Fylingdales",  "lat_deg":  54.36, "lon_deg":   -0.67, "alt_m":  265.0},
        ]

    # -- 4. State vectors for top 20 (synchronous UDLQuery per sat) ----------------
    print(f"\nQuerying latest state vectors for top 20 objects in last {fitspan} days...")
    sv_rows = []
    for sat_no in top20_ids:
        try:
            sv_df = UDLQuery(UDL_TOKEN, "statevector", {
                "satNo": str(int(sat_no)),
                "epoch": time_window,
                "uct": "false",
                "dataMode": "REAL",
                "sort": "epoch,DESC",
                "maxResults": "1",
            })
            if sv_df.empty:
                print(f"  WARN: no state vector for NORAD {sat_no}")
                continue
            sv_rows.append(sv_df.iloc[0])
        except Exception as e:
            print(f"  WARN: state vector query failed for NORAD {sat_no} ({e})")

    state_vectors = pd.DataFrame(sv_rows).reset_index(drop=True)
    if not state_vectors.empty and "epoch" in state_vectors.columns:
        state_vectors["epoch"] = pd.to_datetime(
            state_vectors["epoch"].astype(str).str.replace("Z", ""),
            format="mixed",
            utc=True,
        )

    print(f"\nFinal: {len(state_vectors)} satellites with state vectors")
    print(f"Observatories: {[o['name'] for o in observatories]}")

    state_vectors.to_csv("UDL_Data.csv", index=False)
    pd.DataFrame(observatories).to_csv("UDL_Data.csv", mode='a', header=True, index=False)
    print("Data saved to UDL_Data.csv")

In [22]:
# -- Orekit: State-Vector Initial States + Two-Body Keplerian Propagation -------
from org.orekit.bodies import GeodeticPoint
from org.orekit.frames import TopocentricFrame
from org.orekit.orbits import CartesianOrbit
from org.orekit.utils import PVCoordinates
from org.hipparchus.geometry.euclidean.threed import Vector3D
from org.orekit.propagation.analytical import KeplerianPropagator

# -- 1. Build Keplerian propagators from latest Cartesian state vectors ----------
real_propagators = []
real_sat_nos     = []
real_epochs      = []

required_cols = ["xpos", "ypos", "zpos", "xvel", "yvel", "zvel", "satNo", "epoch"]
missing_cols = [c for c in required_cols if c not in state_vectors.columns]
if missing_cols:
    raise ValueError(f"state_vectors is missing required columns: {missing_cols}")

for _, row in state_vectors.iterrows():
    try:
        # UDL state vectors are typically in km and km/s.
        pos = Vector3D(float(row["xpos"]) * 1e3, float(row["ypos"]) * 1e3, float(row["zpos"]) * 1e3)
        vel = Vector3D(float(row["xvel"]) * 1e3, float(row["yvel"]) * 1e3, float(row["zvel"]) * 1e3)

        epoch_utc = pd.to_datetime(row["epoch"], utc=True).to_pydatetime()
        epoch_abs = AbsoluteDate(
            epoch_utc.year,
            epoch_utc.month,
            epoch_utc.day,
            epoch_utc.hour,
            epoch_utc.minute,
            float(epoch_utc.second + epoch_utc.microsecond / 1e6),
            TimeScalesFactory.getUTC(),
        )

        orbit = CartesianOrbit(
            PVCoordinates(pos, vel),
            inertial_frame,
            epoch_abs,
            Constants.WGS84_EARTH_MU,
        )

        real_propagators.append(KeplerianPropagator(orbit))
        real_sat_nos.append(int(row["satNo"]))
        real_epochs.append(epoch_abs)
    except Exception as e:
        print(f"  Propagator init failed for NORAD {row.get('satNo', 'UNKNOWN')}: {e}")

N_REAL = len(real_propagators)
print(f"Created {N_REAL} Keplerian propagators for: {real_sat_nos}")

# -- 2. Topocentric frames for each observatory ---------------------------------
obs_topo_frames = []
for obs in observatories:
    gp = GeodeticPoint(
        math.radians(float(obs["lat_deg"])),
        math.radians(float(obs["lon_deg"])),
        float(obs["alt_m"]),
    )
    obs_topo_frames.append(TopocentricFrame(earth, gp, obs["name"]))
print(f"Created {len(obs_topo_frames)} topocentric frames")

# -- 3. Simulation time: now -> +1 day at 60-s cadence --------------------------
utc_ts      = TimeScalesFactory.getUTC()
now_utc     = datetime.datetime.utcnow()
sim_start   = AbsoluteDate(
    now_utc.year, now_utc.month, now_utc.day,
    now_utc.hour, now_utc.minute,
    float(now_utc.second + now_utc.microsecond / 1e6), utc_ts
 )
DT_OBS      = 60.0
DUR_OBS     = 86400.0
N_STEPS_OBS = int(DUR_OBS / DT_OBS)
print(f"\nSimulation: {DUR_OBS/3600:.0f} h at {DT_OBS:.0f} s cadence "
      f"({N_STEPS_OBS} steps x {N_REAL} sats x {len(obs_topo_frames)} obs)")
print(f"Start: {now_utc.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print("Running... (may take a few minutes)")

# -- 4. Observation loop ---------------------------------------------------------
records_real = []
for step in range(N_STEPS_OBS):
    t = sim_start.shiftedBy(step * DT_OBS)
    for oi, (obs_f, obs_info) in enumerate(zip(obs_topo_frames, observatories)):
        for si, prop in enumerate(real_propagators):
            try:
                state = prop.propagate(t)
                pos   = state.getPVCoordinates().getPosition()
                frm   = state.getFrame()
                el_d  = math.degrees(obs_f.getElevation(pos, frm, t))
                if el_d <= 0.0:
                    continue
                az_d  = math.degrees(obs_f.getAzimuth(pos, frm, t))
                records_real.append({
                    "time_h":   step * DT_OBS / 3600.0,
                    "obs_idx":  oi,
                    "obs_name": obs_info["name"],
                    "sat_idx":  si,
                    "sat_no":   real_sat_nos[si],
                    "az_deg":   az_d,
                    "el_deg":   el_d,
                })
            except Exception:
                pass

df_real = pd.DataFrame(records_real)
print(f"\nTotal visible observations: {len(df_real):,}")
if not df_real.empty:
    print(df_real.groupby("obs_name")["sat_no"].nunique()
          .rename("distinct satellites visible").to_string())
else:
    print("No visible observations found in the simulated window.")

  Propagator init failed for NORAD nan: Unknown datetime string format, unable to parse: lon_deg, at position 0
Created 8 Keplerian propagators for: [39741, 35752, 40105, 67433, 30323, 24748, 62186, 65107]
Created 3 topocentric frames

Simulation: 24 h at 60 s cadence (1440 steps x 8 sats x 3 obs)
Start: 2026-05-04 02:23:09 UTC
Running... (may take a few minutes)


C:\Users\Louis\AppData\Local\Temp\ipykernel_13424\305870785.py:65: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_utc     = datetime.datetime.utcnow()



Total visible observations: 6,848
obs_name
15SPSS        7
OPTICALX      6
TransAstra    6


In [24]:
# -- Track Analysis: group successive observations into tracks -------------------
import numpy as np
import pandas as pd

# A new track starts if time gap between consecutive observations exceeds this.
# Tune this value to match your definition of a "short timespan".
TRACK_GAP_S = 5 * DT_OBS

if df_real.empty:
    track_summary = pd.DataFrame()
    print("No observations available; track summary is empty.")
else:
    df_tr = df_real.copy()
    df_tr = df_tr.sort_values(["obs_name", "sat_no", "time_h"]).reset_index(drop=True)

    # Convert time_h to seconds for robust duration/gap calculations.
    df_tr["time_s"] = df_tr["time_h"] * 3600.0

    # Gap to previous observation within the same (observatory, satellite) group.
    group_keys = ["obs_name", "sat_no"]
    prev_time_s = df_tr.groupby(group_keys)["time_s"].shift(1)
    gap_s = df_tr["time_s"] - prev_time_s

    # Start a new track if first sample in group or if gap exceeds threshold.
    is_new_track = prev_time_s.isna() | (gap_s > TRACK_GAP_S)
    df_tr["track_id"] = is_new_track.cumsum().astype(int)

    track_summary = (
        df_tr.groupby(["track_id", "obs_name", "sat_no"], as_index=False)
            .agg(
                n_obs=("time_s", "size"),
                start_time_s=("time_s", "min"),
                end_time_s=("time_s", "max"),
            )
    )
    track_summary["duration_s"] = track_summary["end_time_s"] - track_summary["start_time_s"]
    track_summary["start_time_h"] = track_summary["start_time_s"] / 3600.0
    track_summary["end_time_h"] = track_summary["end_time_s"] / 3600.0

    # Reorder columns for readability.
    track_summary = track_summary[[
        "track_id", "obs_name", "sat_no", "n_obs",
        "duration_s", "start_time_h", "end_time_h"
    ]]

    print(f"Track gap threshold: {TRACK_GAP_S:.0f} s")
    print(f"Total tracks found : {len(track_summary):,}")
    print(f"Mean obs/track     : {track_summary['n_obs'].mean():.2f}")
    print(f"Mean duration      : {track_summary['duration_s'].mean():.1f} s")

    print("\nTop 10 longest tracks by duration:")
    print(
        track_summary.sort_values(["duration_s", "n_obs"], ascending=False)
        .head(10)
        .to_string(index=False)
    )

# 'track_summary' now contains one row per detected track.

Track gap threshold: 300 s
Total tracks found : 87
Mean obs/track     : 78.71
Mean duration      : 4662.8 s

Top 10 longest tracks by duration:
 track_id   obs_name  sat_no  n_obs  duration_s  start_time_h  end_time_h
        1     15SPSS   30323    724     43380.0      0.800000   12.850000
        6     15SPSS   40105    543     32520.0      0.133333    9.166667
       64 TransAstra   40105    447     26760.0      1.950000    9.383333
       36   OPTICALX   40105    442     26460.0      1.950000    9.300000
        2     15SPSS   35752    369     22080.0      6.100000   12.233333
        4     15SPSS   39741    369     22080.0      2.216667    8.350000
       61 TransAstra   35752    358     21420.0     13.833333   19.783333
       63 TransAstra   39741    356     21300.0     10.033333   15.950000
       33   OPTICALX   35752    355     21240.0     13.933333   19.833333
       35   OPTICALX   39741    353     21120.0     10.133333   16.000000


In [25]:
# ── Multi-Observatory Observation Plots ──────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc

N_OBS_LOC    = len(observatories)
palette_real = pc.qualitative.Dark24
all_sat_nos  = sorted(df_real["sat_no"].unique())

# ── 1. Sky plots (Az/El) — one polar per observatory ─────────────────────────
fig_sky = make_subplots(
    rows=1, cols=N_OBS_LOC,
    specs=[[{"type": "polar"}] * N_OBS_LOC],
    subplot_titles=[o["name"] for o in observatories],
    horizontal_spacing=0.04,
)
polar_cfg = dict(
    angularaxis=dict(direction="clockwise", rotation=90,
                     tickvals=[0, 90, 180, 270], ticktext=["N","E","S","W"],
                     gridcolor="rgba(100,100,100,0.4)"),
    radialaxis=dict(range=[0, 90], tickvals=[30, 60, 90],
                    ticktext=["60°","30°","0°"], tickfont=dict(size=9)),
)

for oi in range(N_OBS_LOC):
    sub = df_real[df_real["obs_idx"] == oi]
    polar_ref = "polar" if oi == 0 else f"polar{oi + 1}"
    for rank, sat_no in enumerate(all_sat_nos):
        grp = sub[sub["sat_no"] == sat_no]
        if grp.empty:
            continue
        fig_sky.add_trace(go.Scatterpolar(
            r=90 - grp["el_deg"], theta=grp["az_deg"],
            mode="markers", marker=dict(size=3, color=palette_real[rank % len(palette_real)]),
            name=f"NORAD {sat_no}", legendgroup=str(sat_no),
            showlegend=(oi == 0),
            subplot=polar_ref,
        ), row=1, col=oi + 1)

for i in range(1, N_OBS_LOC + 1):
    key = "polar" if i == 1 else f"polar{i}"
    fig_sky.update_layout(**{key: polar_cfg})

fig_sky.update_layout(
    title_text="Sky Coverage per Observatory — 1 Day (zenith=centre, horizon=edge)",
    height=520,
    legend=dict(title="Satellite", font=dict(size=9), tracegroupgap=2),
)
fig_sky.show(renderer="browser")

# ── 2. Elevation vs Time — stacked by observatory ────────────────────────────
fig_el = make_subplots(
    rows=N_OBS_LOC, cols=1,
    subplot_titles=[o["name"] for o in observatories],
    shared_xaxes=True,
    vertical_spacing=0.06,
)
for oi in range(N_OBS_LOC):
    sub = df_real[df_real["obs_idx"] == oi]
    for rank, sat_no in enumerate(all_sat_nos):
        grp = sub[sub["sat_no"] == sat_no]
        if grp.empty:
            continue
        fig_el.add_trace(go.Scatter(
            x=grp["time_h"], y=grp["el_deg"],
            mode="markers", marker=dict(size=2, color=palette_real[rank % len(palette_real)]),
            name=f"NORAD {sat_no}", legendgroup=str(sat_no),
            showlegend=(oi == 0),
        ), row=oi + 1, col=1)
    fig_el.update_yaxes(title_text="El [°]", range=[0, 90], row=oi + 1, col=1)

fig_el.update_xaxes(title_text="Time [hours]", range=[0, 24], row=N_OBS_LOC, col=1)
fig_el.update_layout(
    title_text="Elevation vs Time per Observatory — 1 Day",
    height=180 * N_OBS_LOC + 80,
    legend=dict(title="Satellite", font=dict(size=9), tracegroupgap=2),
)
fig_el.show(renderer="browser")

In [26]:
# -- Scheduling metric: select one satellite per observatory per time step --
# Switch SCHEDULING_STRATEGY to change which metric is active.
#   'priority'     : static priority dict - lowest value wins
#   'arclength'    : least total observed arclength wins (promotes equal coverage)
#   'random_dwell' : observe a random satellite for MIN_OBS_TIME_S (or until it
#                    leaves the FOV), then pick a new random target

import random

SCHEDULING_STRATEGY = 'random_dwell'   # 'priority' | 'arclength' | 'random_dwell'

# Minimum time to observe a target before switching (applies to all strategies).
# Prevents rapid oscillation when two objects have similar metric scores.
# MIN_OBS_TIME_FRAMES is derived in the animation cell from ANIM_DT_R.
MIN_OBS_TIME_S = 15 * 60   # seconds  <-- change this to adjust the dwell floor

# Arbitrary constant priority weights keyed by NORAD ID (lower = observed first).
# Built from real_sat_nos so keys always match the current satellite set.
# Edit the weights list to change individual priorities.
SATELLITE_PRIORITIES = {
    norad: w
    for norad, w in zip(
        real_sat_nos,
        [3.2, 1.5, 4.8, 2.1, 7.3, 0.9, 5.5, 3.7, 6.1, 2.8,
         8.0, 1.2, 4.4, 6.9, 0.5, 7.7, 3.3, 5.0, 2.6, 9.1],
    )
}

def scheduling_metric(vis_sat_indices, metric_dict=None, prev_selected=-1, dwell_frames=0):
    """Select one satellite from those currently in the field of view.

    Strategies
    ----------
    'priority'     : pick the satellite with the lowest static priority value.
    'arclength'    : pick the satellite with the least accumulated observed
                     arclength (degrees, summed across all observatories).
    'random_dwell' : observe a randomly chosen satellite for MIN_OBS_TIME_S or
                     until it leaves the FOV, then select a new random target.

    All strategies respect MIN_OBS_TIME_FRAMES: the current target is held for
    at least that many frames (provided it stays visible) before the metric is
    re-evaluated, preventing rapid oscillation on closely-scored objects.

    Parameters
    ----------
    vis_sat_indices : list[int]
        Indices (into real_sat_nos) of satellites currently above the horizon.
    metric_dict : dict[int, float] | None
        Mapping of NORAD ID -> score.  Pass SATELLITE_PRIORITIES for the
        'priority' strategy, or obs_arclength for the 'arclength' strategy.
    prev_selected : int
        Satellite index selected at the previous frame for this observatory.
    dwell_frames : int
        Number of consecutive frames the current target has been observed.

    Returns
    -------
    int : selected satellite index, or -1 if none are visible.
    """
    if len(vis_sat_indices) == 0:
        return -1

    still_visible = prev_selected in vis_sat_indices
    min_time_met  = dwell_frames >= MIN_OBS_TIME_FRAMES

    # Hold current target if min dwell time has not yet been reached
    if still_visible and not min_time_met:
        return prev_selected

    if SCHEDULING_STRATEGY == 'random_dwell':
        # Min time met (or target set): pick a different random target if possible
        candidates = [s for s in vis_sat_indices if s != prev_selected] or vis_sat_indices
        return random.choice(candidates)
    if SCHEDULING_STRATEGY == 'arclength' and metric_dict is not None:
        return min(vis_sat_indices, key=lambda s: metric_dict[real_sat_nos[s]])
    return min(vis_sat_indices, key=lambda s: SATELLITE_PRIORITIES[real_sat_nos[s]])


In [27]:
# -- 3D Animation: Real Objects (EME2000) + Separate Sky Plots ------------------
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import numpy as np

ANIM_DT_R    = 60.0                       # 1-min steps
ANIM_DUR_R   = 4 * 3600.0                 # 4-hour animation window
N_ANIM_R          = int(ANIM_DUR_R / ANIM_DT_R)
MIN_OBS_TIME_FRAMES = int(MIN_OBS_TIME_S / ANIM_DT_R)  # derived from scheduling cell
Re_km_r      = Constants.WGS84_EARTH_EQUATORIAL_RADIUS / 1e3
TRAIL_R      = 12
palette_anim = pc.qualitative.Dark24
OBS_COLORS   = ['yellow', 'cyan', 'magenta', 'orange', 'lime']
N_OBS_ANIM   = len(obs_topo_frames)
N_SKY        = min(2, N_OBS_ANIM)

if N_SKY < 2:
    print('Only one observatory found; showing one sky plot.')

# -- 1. ITRF -> EME2000 rotation matrices --------------------------------------
print('Computing rotation matrices...')
rot_r = np.zeros((N_ANIM_R, 3, 3))
for f in range(N_ANIM_R):
    t  = sim_start.shiftedBy(f * ANIM_DT_R)
    Rm = itrf.getTransformTo(inertial_frame, t).getRotation().getMatrix()
    rot_r[f] = [[Rm[i][j] for j in range(3)] for i in range(3)]

# -- 2. All observatories ECEF -> EME2000 per frame ----------------------------
all_obs_eme_r = []
for obs in observatories:
    obs_r_km = Re_km_r + obs['alt_m'] / 1e3
    ecef     = np.array([
        obs_r_km * np.cos(math.radians(obs['lat_deg'])) * np.cos(math.radians(obs['lon_deg'])),
        obs_r_km * np.cos(math.radians(obs['lat_deg'])) * np.sin(math.radians(obs['lon_deg'])),
        obs_r_km * np.sin(math.radians(obs['lat_deg'])),
    ])
    all_obs_eme_r.append(np.einsum('fij,j->fi', rot_r, ecef))

# -- 3. Earth grid ---------------------------------------------------------------
N_PT_R   = 100
segs_r   = []
for ld in np.arange(0, 360, 30):
    lats = np.linspace(-np.pi/2, np.pi/2, N_PT_R)
    lo   = math.radians(ld)
    seg  = np.stack([Re_km_r*np.cos(lats)*np.cos(lo),
                     Re_km_r*np.cos(lats)*np.sin(lo),
                     Re_km_r*np.sin(lats)], axis=1)
    segs_r += [seg, np.full((1, 3), np.nan)]
for ld in np.arange(-60, 61, 30):
    lons = np.linspace(0, 2*np.pi, N_PT_R)
    la   = math.radians(ld)
    seg  = np.stack([Re_km_r*np.cos(la)*np.cos(lons),
                     Re_km_r*np.cos(la)*np.sin(lons),
                     Re_km_r*np.sin(la)*np.ones(N_PT_R)], axis=1)
    segs_r += [seg, np.full((1, 3), np.nan)]
grid_base_r  = np.vstack(segs_r)
valid_r      = ~np.isnan(grid_base_r[:, 0])
grid_eme_r   = np.full((N_ANIM_R, len(grid_base_r), 3), np.nan)
grid_eme_r[:, valid_r, :] = np.einsum('fij,kj->fki', rot_r, grid_base_r[valid_r])

# -- 4. Satellite positions + visibility/az-el from all observatories ----------
print(f'Pre-computing {N_ANIM_R} frames x {N_REAL} real satellites...')
pos_r    = np.zeros((N_ANIM_R, N_REAL, 3))
vis_r    = np.zeros((N_ANIM_R, N_OBS_ANIM, N_REAL), dtype=bool)
az_r     = np.zeros((N_ANIM_R, N_OBS_ANIM, N_REAL))
el_r     = np.zeros((N_ANIM_R, N_OBS_ANIM, N_REAL))

for f in range(N_ANIM_R):
    t = sim_start.shiftedBy(f * ANIM_DT_R)
    for s, prop in enumerate(real_propagators):
        try:
            state  = prop.propagate(t)
            frm    = state.getFrame()
            pos_tm = state.getPVCoordinates().getPosition()

            pos_em = frm.getTransformTo(inertial_frame, t).transformPosition(pos_tm)
            pos_r[f, s] = [pos_em.getX()/1e3, pos_em.getY()/1e3, pos_em.getZ()/1e3]

            for oi, obs_f in enumerate(obs_topo_frames):
                el_d = math.degrees(obs_f.getElevation(pos_tm, frm, t))
                el_r[f, oi, s] = el_d
                if el_d > 0.0:
                    vis_r[f, oi, s] = True
                    az_r[f, oi, s]  = math.degrees(obs_f.getAzimuth(pos_tm, frm, t))
        except Exception:
            pass
print('Done.')

# -- 5. Apply scheduling metric: select one satellite per observatory per frame --
# obs_arclength: {NORAD_ID: total_degrees_observed} across all observatories combined.
# Updated sequentially so later frames see the accumulated totals from earlier ones.
selected_r    = np.full((N_ANIM_R, N_OBS_ANIM), -1, dtype=int)
obs_arclength = {norad: 0.0 for norad in real_sat_nos}  # keyed by NORAD ID
dwell_count   = np.zeros(N_OBS_ANIM, dtype=int)         # frames on current target per obs

for f in range(N_ANIM_R):
    # (a) Select satellite for each observatory
    for oi in range(N_OBS_ANIM):
        vis_sats = np.where(vis_r[f, oi])[0].tolist()
        prev = int(selected_r[f - 1, oi]) if f > 0 else -1
        selected_r[f, oi] = scheduling_metric(
            vis_sats, obs_arclength,
            prev_selected=prev, dwell_frames=dwell_count[oi],
        )
        # Track consecutive frames on current target
        if selected_r[f, oi] == prev:
            dwell_count[oi] += 1
        else:
            dwell_count[oi] = 0

    # (b) Update arclength: for each selected satellite, add the angular arc it
    #     traversed across the sky between the previous and current frame.
    #     Uses spherical law of cosines in Az/El space.
    if f > 0:
        for oi in range(N_OBS_ANIM):
            s = selected_r[f, oi]
            if s < 0 or not vis_r[f - 1, oi, s]:
                continue   # not visible at previous frame; no arc to accumulate
            el1 = np.radians(el_r[f - 1, oi, s])
            el2 = np.radians(el_r[f,     oi, s])
            az1 = np.radians(az_r[f - 1, oi, s])
            az2 = np.radians(az_r[f,     oi, s])
            arc = np.degrees(np.arccos(np.clip(
                np.sin(el1) * np.sin(el2)
                + np.cos(el1) * np.cos(el2) * np.cos(az1 - az2),
                -1.0, 1.0,
            )))
            obs_arclength[real_sat_nos[s]] += arc

# Lock scene extent so Earth/sat scale stays constant frame-to-frame
r_norm_max = np.nanmax(np.linalg.norm(pos_r, axis=2)) if pos_r.size else Re_km_r
AXIS_LIM_KM = max(45000.0, 1.10 * float(r_norm_max))

# -- 6. Static sphere + equator --------------------------------------------------
N_sph_r = 60
ps_r    = np.linspace(0, np.pi, N_sph_r)
ts_r    = np.linspace(0, 2*np.pi, N_sph_r)
x_er    = Re_km_r * np.outer(np.sin(ps_r), np.cos(ts_r))
y_er    = Re_km_r * np.outer(np.sin(ps_r), np.sin(ts_r))
z_er    = Re_km_r * np.outer(np.cos(ps_r), np.ones(N_sph_r))
eq_t_r  = np.linspace(0, 2*np.pi, 361)
eq_xr, eq_yr, eq_zr = Re_km_r*np.cos(eq_t_r), Re_km_r*np.sin(eq_t_r), np.zeros(361)

# -- 7. Per-frame helpers --------------------------------------------------------
def sat_trace_r(f):
    vis_any = vis_r[f].any(axis=0)
    colors = [palette_anim[s % len(palette_anim)] for s in range(N_REAL)]
    sizes  = [9 if vis_any[s] else 5 for s in range(N_REAL)]
    edges  = ['white' if vis_any[s] else 'rgba(0,0,0,0)' for s in range(N_REAL)]
    labels = [
        f'NORAD {real_sat_nos[s]}<br>{"VISIBLE (>=1 obs)" if vis_any[s] else "not visible"}'
        for s in range(N_REAL)
    ]
    return dict(
        x=pos_r[f,:,0], y=pos_r[f,:,1], z=pos_r[f,:,2],
        marker=dict(color=colors, size=sizes, line=dict(color=edges, width=1.5)),
        text=labels, hovertemplate='%{text}<extra></extra>'
    )

def trail_trace_r(f):
    f0 = max(0, f - TRAIL_R)
    xs, ys, zs = [], [], []
    for s in range(N_REAL):
        xs += list(pos_r[f0:f+1, s, 0]) + [None]
        ys += list(pos_r[f0:f+1, s, 1]) + [None]
        zs += list(pos_r[f0:f+1, s, 2]) + [None]
    return dict(x=xs, y=ys, z=zs)

def sky_curr_r(f, oi):
    vis_sats = np.where(vis_r[f, oi])[0]
    if len(vis_sats) == 0:
        return dict(r=[], theta=[], marker=dict(color=[], size=10), text=[])
    labels = [
        f"{observatories[oi]['name']}<br>NORAD {real_sat_nos[s]}<br>El:{el_r[f,oi,s]:.1f} deg  Az:{az_r[f,oi,s]:.1f} deg"
        for s in vis_sats
    ]
    return dict(
        r=list(90 - el_r[f, oi, vis_sats]),
        theta=list(az_r[f, oi, vis_sats]),
        marker=dict(color=[palette_anim[s % len(palette_anim)] for s in vis_sats], size=9),
        text=labels, hovertemplate='%{text}<extra></extra>'
    )

def sky_trail_r(f, oi):
    f0 = max(0, f - TRAIL_R)
    r_a, th_a, col_a = [], [], []
    for s in range(N_REAL):
        if not vis_r[f, oi, s]:
            continue
        col = palette_anim[s % len(palette_anim)]
        for ff in range(f0, f):
            if vis_r[ff, oi, s]:
                r_a.append(90 - el_r[ff, oi, s])
                th_a.append(az_r[ff, oi, s])
                col_a.append(col)
    if not r_a:
        return dict(r=[], theta=[], marker=dict(color='rgba(0,0,0,0)', size=4, opacity=0.0))
    return dict(r=r_a, theta=th_a, marker=dict(color=col_a, size=4, opacity=0.30))

def sky_selected_r(f, oi):
    """Golden diamond marker for the satellite currently being observed."""
    s = selected_r[f, oi]
    if s < 0:
        return dict(r=[], theta=[], marker=dict(
            color='rgba(0,0,0,0)', size=16, symbol='diamond',
            line=dict(color='gold', width=2)), text=[])
    label = (
        f'OBSERVING: NORAD {real_sat_nos[s]}'
        f'<br>El:{el_r[f,oi,s]:.1f}°  Az:{az_r[f,oi,s]:.1f}°'
        f'<br>Priority: {SATELLITE_PRIORITIES[real_sat_nos[s]]:.1f}'
    )
    return dict(
        r=[90 - el_r[f, oi, s]],
        theta=[az_r[f, oi, s]],
        marker=dict(color='rgba(0,0,0,0)', size=16, symbol='diamond',
                    line=dict(color='gold', width=2)),
        text=[label],
        hovertemplate='%{text}<extra></extra>',
    )

# -- 8. Trace layout -------------------------------------------------------------
col_widths = [0.60] + ([0.40 / N_SKY] * N_SKY)
specs_row = [{"type": "scene"}] + ([{"type": "polar"}] * N_SKY)
titles = [f'Real Objects - EME2000 ({N_REAL} sats)'] + [f'Sky Plot - {observatories[i]["name"]}' for i in range(N_SKY)]
fig_r = make_subplots(
    rows=1, cols=1 + N_SKY,
    column_widths=col_widths,
    specs=[specs_row],
    subplot_titles=titles,
 )

# Static traces
fig_r.add_trace(go.Surface(
    x=x_er, y=y_er, z=z_er,
    colorscale=[[0,'#1a4fa0'],[1,'#2979c8']],
    showscale=False, opacity=0.85, hoverinfo='skip', name='Earth'
), row=1, col=1)
fig_r.add_trace(go.Scatter3d(
    x=eq_xr, y=eq_yr, z=eq_zr, mode='lines',
    line=dict(color='rgba(255,255,255,0.5)', width=1),
    hoverinfo='skip', showlegend=False
), row=1, col=1)

# Animated traces start here
fig_r.add_trace(go.Scatter3d(
    x=grid_eme_r[0,:,0], y=grid_eme_r[0,:,1], z=grid_eme_r[0,:,2],
    mode='lines', line=dict(color='rgba(255,255,255,0.2)', width=1),
    hoverinfo='skip', showlegend=False
), row=1, col=1)

for oi, (oe, obs_info) in enumerate(zip(all_obs_eme_r, observatories)):
    fig_r.add_trace(go.Scatter3d(
        x=[oe[0,0]], y=[oe[0,1]], z=[oe[0,2]], mode='markers+text',
        marker=dict(size=7, color=OBS_COLORS[oi % len(OBS_COLORS)], symbol='diamond'),
        text=[obs_info['name']], textposition='top center',
        textfont=dict(color=OBS_COLORS[oi % len(OBS_COLORS)], size=10),
        name=obs_info['name']
    ), row=1, col=1)

tt0 = trail_trace_r(0)
fig_r.add_trace(go.Scatter3d(
    x=tt0['x'], y=tt0['y'], z=tt0['z'], mode='lines',
    line=dict(color='rgba(255,255,255,0.15)', width=1),
    hoverinfo='skip', showlegend=False
), row=1, col=1)
st0 = sat_trace_r(0)
fig_r.add_trace(go.Scatter3d(mode='markers', **st0, name='Satellites'), row=1, col=1)

for oi in range(N_SKY):
    fig_r.add_trace(go.Scatterpolar(mode='markers', **sky_trail_r(0, oi), hoverinfo='skip', showlegend=False), row=1, col=2 + oi)
    fig_r.add_trace(go.Scatterpolar(mode='markers', **sky_curr_r(0, oi), showlegend=False), row=1, col=2 + oi)
    fig_r.add_trace(go.Scatterpolar(
        mode='markers', **sky_selected_r(0, oi),
        name=f'Observing ({observatories[oi]["name"]})', showlegend=False,
    ), row=1, col=2 + oi)

# Indices of animated traces
first_anim_idx = 2
last_anim_idx = len(fig_r.data) - 1
anim_traces_r = list(range(first_anim_idx, last_anim_idx + 1))

# -- 9. Animation frames ---------------------------------------------------------
frames_r = []
for f in range(N_ANIM_R):
    t_hr  = f * ANIM_DT_R / 3600.0
    n_vis = int(vis_r[f].any(axis=0).sum())
    frame_data = [
        go.Scatter3d(x=grid_eme_r[f,:,0], y=grid_eme_r[f,:,1], z=grid_eme_r[f,:,2], mode='lines'),
    ]
    for oi, oe in enumerate(all_obs_eme_r):
        frame_data.append(go.Scatter3d(
            x=[oe[f,0]], y=[oe[f,1]], z=[oe[f,2]], mode='markers+text',
            marker=dict(size=7, color=OBS_COLORS[oi % len(OBS_COLORS)], symbol='diamond'),
            text=[observatories[oi]['name']], textposition='top center',
            textfont=dict(color=OBS_COLORS[oi % len(OBS_COLORS)], size=10),
        ))
    tt = trail_trace_r(f)
    st = sat_trace_r(f)
    frame_data += [
        go.Scatter3d(mode='lines', **tt),
        go.Scatter3d(mode='markers', **st),
    ]
    for oi in range(N_SKY):
        frame_data.append(go.Scatterpolar(mode='markers', **sky_trail_r(f, oi)))
        frame_data.append(go.Scatterpolar(mode='markers', **sky_curr_r(f, oi)))
        frame_data.append(go.Scatterpolar(mode='markers', **sky_selected_r(f, oi)))

    frames_r.append(go.Frame(
        data=frame_data, traces=anim_traces_r, name=str(f),
        layout=go.Layout(title_text=(
            f'Real Objects (EME2000) - T+{t_hr:.2f} h  |  '
            f'{n_vis} visible from at least one observatory'
        )),
    ))
fig_r.frames = frames_r

# -- 10. Layout (fixed scene range/aspect) --------------------------------------
fig_r.update_layout(
    title_text=(f'Real Objects (EME2000) - T+0.00 h  |  '
                f'{int(vis_r[0].any(axis=0).sum())} visible from at least one observatory'),
    paper_bgcolor='black', font_color='white', height=750,
    uirevision='lock-scale',
    scene=dict(
        bgcolor='black',
        xaxis=dict(visible=False, autorange=False, range=[-AXIS_LIM_KM, AXIS_LIM_KM]),
        yaxis=dict(visible=False, autorange=False, range=[-AXIS_LIM_KM, AXIS_LIM_KM]),
        zaxis=dict(visible=False, autorange=False, range=[-AXIS_LIM_KM, AXIS_LIM_KM]),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1),
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
    ),
    legend=dict(x=0.01, y=0.95, font=dict(size=10)),
    updatemenus=[dict(
        type='buttons', showactive=False, x=0.05, y=0.05,
        xanchor='left', yanchor='bottom',
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, dict(frame=dict(duration=400, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0), mode='immediate')]),
        ],
    )],
    sliders=[dict(
        steps=[dict(args=[[str(f)], dict(frame=dict(duration=0), mode='immediate')],
                    label=f'{f*ANIM_DT_R/3600:.1f}h', method='animate')
               for f in range(N_ANIM_R)],
        transition=dict(duration=0), x=0.1, len=0.85, y=0.02,
        currentvalue=dict(prefix='Time: ', visible=True, xanchor='center',
                          font=dict(color='white')),
        font=dict(color='white'),
    )],
)

polar_cfg = dict(
    bgcolor='#05050f',
    angularaxis=dict(direction='clockwise', rotation=90,
                     tickvals=[0, 90, 180, 270], ticktext=['N', 'E', 'S', 'W'],
                     gridcolor='rgba(255,255,255,0.15)',
                     linecolor='rgba(255,255,255,0.3)'),
    radialaxis=dict(range=[0, 90], tickvals=[30, 60, 90],
                    ticktext=['60 deg', '30 deg', '0 deg'],
                    tickfont=dict(size=9, color='rgba(255,255,255,0.5)'),
                    gridcolor='rgba(255,255,255,0.15)',
                    linecolor='rgba(255,255,255,0.15)'),
)
for i in range(N_SKY):
    key = 'polar' if i == 0 else f'polar{i+1}'
    fig_r.update_layout(**{key: polar_cfg})

fig_r.show(renderer='browser')


Computing rotation matrices...
Pre-computing 240 frames x 8 real satellites...
Done.


In [28]:
# -- Semi-major axis from state vectors -----------------------------------------
import numpy as np
import pandas as pd

if state_vectors.empty:
    sma_df = pd.DataFrame()
    print("state_vectors is empty; no semi-major axes to compute.")
else:
    # Convert km and km/s to SI for consistent energy units.
    r_vec_m = state_vectors[["xpos", "ypos", "zpos"]].astype(float).to_numpy() * 1e3
    v_vec_mps = state_vectors[["xvel", "yvel", "zvel"]].astype(float).to_numpy() * 1e3

    r_mag = np.linalg.norm(r_vec_m, axis=1)
    v_mag = np.linalg.norm(v_vec_mps, axis=1)

    # Specific orbital energy: epsilon = v^2/2 - mu/r
    # Semi-major axis: a = -mu / (2*epsilon) for bound orbits.
    eps = 0.5 * v_mag**2 - mu / r_mag
    with np.errstate(divide="ignore", invalid="ignore"):
        a_m = -mu / (2.0 * eps)

    sma_df = pd.DataFrame({
        "sat_no": state_vectors["satNo"].astype(int),
        "epoch": state_vectors["epoch"],
        "specific_energy_m2_s2": eps,
        "semi_major_axis_km": a_m / 1e3,
    })

    # Keep physically bound two-body solutions (a > 0).
    sma_df["is_bound"] = sma_df["semi_major_axis_km"] > 0.0
    sma_df = sma_df.sort_values(["semi_major_axis_km", "sat_no"]).reset_index(drop=True)

    print(f"Computed semi-major axis for {len(sma_df)} objects")
    print(sma_df[["sat_no", "semi_major_axis_km", "is_bound"]].head(20).to_string(index=False))

# 'sma_df' contains semi-major axis (km) for each object.

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer